# Verificación de la capa Bronze — Iceberg

Se revisan las tablas Iceberg en `nessie.nyc` tras la ingesta:
referencias Nessie, metadatos, snapshots, particionado y calidad de datos.

In [1]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie

spark = get_spark("verify-bronze-iceberg")
spark.version

26/07/14 16:41:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


'3.5.8'

## 1. Catálogo Nessie — Referencias y namespaces

La ingesta bronze usa WAP: cada mes crea una rama efímera, escribe, valida y
hace merge a `bronze`. Tras el último mes de cada año se crea un tag de auditoría.

In [2]:
# Referencias Nessie (ramas y tags)
print("Referencias Nessie:\n")
nessie.print_references(spark)

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
branches = refs_df.filter(refs_df[type_col].ilike("branch")).count()
tags = refs_df.filter(refs_df[type_col].ilike("tag")).count()
print(f"\n  Ramas: {branches}  |  Tags: {tags}")

Referencias Nessie:



  * Branch bronze                                   bf8e6494579d
  * Branch main                                     c577394a0346
  * Branch silver                                   8bbc7435661a
    Tag    bronze-taxi_zone_lookup-2026             bf8e6494579d
    Tag    bronze-yellow_taxi-2023                  6d69f7093dc9
    Tag    bronze-yellow_taxi-2024                  1c1fdaf5d2e3
    Tag    bronze-yellow_taxi-2025                  3191b472cb24
    Tag    gold-pickup_demand_zone_hour-2023        8b8ef9c9e815
    Tag    gold-pickup_demand_zone_hour-2024        fbe8ff3bb080
    Tag    gold-pickup_demand_zone_hour-2025        c577394a0346
    Tag    gold-yellow_taxi_trips-2023              ac0fe2b5b662
    Tag    gold-yellow_taxi_trips-2024              3d78ec52a3d4
    Tag    gold-yellow_taxi_trips-2025              197cc46e37ad
    Tag    silver-taxi_zone_lookup-2026             8bbc7435661a
    Tag    silver-yellow_taxi-2023                  c8ec624d8bad
    Tag    silver-yellow_


  Ramas: 3  |  Tags: 14


In [3]:
# Se cambia a la rama bronze para el resto del notebook
nessie.set_spark_branch(spark, "bronze")

# Namespaces disponibles
spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)

+---------+
|namespace|
+---------+
|nyc      |
+---------+



In [4]:
# Tablas en el namespace bronze
spark.sql("SHOW TABLES IN nessie.nyc").show(truncate=False)

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|nyc      |taxi_zone_lookup|false      |
|nyc      |yellow_taxi     |false      |
+---------+----------------+-----------+



## 2. Yellow Taxi — Propiedades y esquema

In [5]:
# Esquema, particionado, propiedades
spark.sql("DESCRIBE EXTENDED nessie.nyc.yellow_taxi").show(250, truncate=False)

+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                                                     |comment|
+-----------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [6]:
# Propiedades Iceberg (versión de formato, gc, etc.)
spark.sql("SHOW TBLPROPERTIES nessie.nyc.yellow_taxi").show(truncate=False)

+------------------------------------------+----------------------------------------------------------------+
|key                                       |value                                                           |
+------------------------------------------+----------------------------------------------------------------+
|current-snapshot-id                       |7819480518084378669                                             |
|format                                    |iceberg/parquet                                                 |
|format-version                            |2                                                               |
|gc.enabled                                |false                                                           |
|nessie.commit.id                          |bf8e6494579df5e79731a1e1e380d123072a16b199617b34b6507288e312061d|
|write.metadata.delete-after-commit.enabled|false                                                           |
|write.par

## 3. Snapshots e historial

In [7]:
# Cada write/overwrite crea un snapshot
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.snapshots").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+--------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                       |summary                                                                                                                      

In [8]:
# Historial (registro de snapshots ordenado por tiempo)
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-07-14 09:48:11.926|6106427666344408779|NULL               |true               |
|2026-07-14 09:48:23.951|862440423282045441 |6106427666344408779|true               |
|2026-07-14 09:48:36.694|1284933862171102643|862440423282045441 |true               |
|2026-07-14 09:48:49.275|2880659764696534364|1284933862171102643|true               |
|2026-07-14 09:48:58.761|6104533909960610743|2880659764696534364|true               |
|2026-07-14 09:49:09.566|3588176452150897526|6104533909960610743|true               |
|2026-07-14 09:49:21.634|367823404095416790 |3588176452150897526|true               |
|2026-07-14 09:49:35.211|3141165920436118132|367823404095416790 |true               |
|2026-07-14 09:49:47.906|7819480518084378669|314116592

## 3b. Crecimiento de registros por tag

Cada tag Nessie captura el estado de `bronze` tras ingestar todos los meses
de un año. Leyendo la tabla en cada tag se ve cómo crece año a año.

In [9]:
# Conteo de registros en cada tag anual para ver el crecimiento incremental
import re

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

tag_rows_df = refs_df.filter(refs_df[type_col].ilike("tag")) \
    .filter(refs_df["name"].rlike(r"^bronze-yellow_taxi-\d{4}$"))
tag_names = [r["name"] for r in tag_rows_df.collect()]

tag_pattern = re.compile(r"bronze-yellow_taxi-(\d{4})")
tag_rows = []
for name in tag_names:
    m = tag_pattern.match(name)
    if m:
        tag_rows.append((name, int(m.group(1))))
tag_rows.sort(key=lambda t: t[1])

growth = []
for tag_name, year in tag_rows:
    nessie.set_spark_branch(spark, tag_name)
    count = spark.table("nessie.nyc.yellow_taxi").count()
    growth.append((tag_name, year, count))

nessie.set_spark_branch(spark, "bronze")

growth_df = spark.createDataFrame(
    [(tag, year, total, total - (growth[i-1][2] if i > 0 else 0))
     for i, (tag, year, total) in enumerate(growth)],
    ["tag", "año", "registros_acumulados", "registros_añadidos"]
)
growth_df.show(truncate=False)

+-----------------------+----+--------------------+------------------+
|tag                    |año |registros_acumulados|registros_añadidos|
+-----------------------+----+--------------------+------------------+
|bronze-yellow_taxi-2023|2023|9384487             |9384487           |
|bronze-yellow_taxi-2024|2024|18939265            |9554778           |
|bronze-yellow_taxi-2025|2025|30137291            |11198026          |
+-----------------------+----+--------------------+------------------+



## 4. Particionado y ficheros

In [10]:
# Especificación de particionado
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.partitions").show(truncate=False)

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{648}    |0      |2964617     |2         |46572601                     |0                           |0                         |0                           |0                         |2026-07-14 09:48:58.761|6104533909960610743     |
|{649}    |0      |3007533     |3         |46917390         

In [11]:
# Ficheros de datos: rutas, tamaños, registros por fichero
files_df = spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.files")
files_df.select(
    "content", "file_path", "file_format",
    "record_count", "file_size_in_bytes"
).show(100, truncate=80)

+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|content|                                                                       file_path|file_format|record_count|file_size_in_bytes|
+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2025-0...|    PARQUET|     4145224|          66128146|
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2025-0...|    PARQUET|           2|              6376|
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2007-1...|    PARQUET|           1|              5908|
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tpep_pickup_datetime_month=2009-0...|    PARQUET|           1|              5908|
|      0|s3a://warehouse/bronze/nyc/yellow_taxi/data/tp

In [12]:
# Resumen: ficheros totales, tamaño, registros
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

files_df.select(
    count("*").alias("ficheros_totales"),
    spark_round(spark_sum("file_size_in_bytes") / 1024 / 1024, 2).alias("tamaño_total_mb"),
    spark_sum("record_count").alias("registros_totales"),
).show()

+----------------+---------------+-----------------+
|ficheros_totales|tamaño_total_mb|registros_totales|
+----------------+---------------+-----------------+
|              43|         452.07|         30137291|
+----------------+---------------+-----------------+



## 5. Manifiestos

In [13]:
# Una entrada por snapshot
spark.sql("SELECT * FROM nessie.nyc.yellow_taxi.manifests").show(truncate=80)

+-------+--------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+----------------------------------+
|content|                                                                            path|length|partition_spec_id|  added_snapshot_id|added_data_files_count|existing_data_files_count|deleted_data_files_count|added_delete_files_count|existing_delete_files_count|deleted_delete_files_count|               partition_summaries|
+-------+--------------------------------------------------------------------------------+------+-----------------+-------------------+----------------------+-------------------------+------------------------+------------------------+---------------------------+--------------------------+----------------------------------+
|      0|s3a://warehouse/

## 6. Calidad de datos

In [14]:
df = spark.table("nessie.nyc.yellow_taxi")
print(f"Registros totales: {df.count():,}")
print(f"Columnas: {len(df.columns)}")
df.printSchema()

Registros totales: 30,137,291
Columnas: 20
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [ ]:
# Registros por año-mes
from pyspark.sql.functions import year, month

df.groupBy(
    year("tpep_pickup_datetime").alias("año"),
    month("tpep_pickup_datetime").alias("mes")
).count().withColumnRenamed("count", "registros") \
 .orderBy("año", "mes") \
 .show(50)

+----+---+---------+
| año|mes|registros|
+----+---+---------+
|2001|  1|        3|
|2002| 12|        6|
|2003|  1|        3|
|2007| 12|        1|
|2008| 12|        9|
|2009|  1|        6|
|2014| 11|        1|
|2022| 10|       11|
|2022| 12|       25|
|2023|  1|  3066726|
|2023|  2|  2914003|
|2023|  3|  3403619|
|2023|  4|       85|
|2023| 12|       10|
|2024|  1|  2964617|
|2024|  2|  3007533|
|2024|  3|  3582607|
|2024|  4|        2|
|2024| 12|       21|
|2025|  1|  3475234|
|2025|  2|  3577542|
|2025|  3|  4145225|
|2025|  4|        2|
+----+---+---------+



In [16]:
# Nulos por columna
from pyspark.sql.functions import col as c, sum as spark_sum

null_counts = df.select([
    spark_sum(c(col).isNull().cast("int")).alias(col)
    for col in df.columns
])
null_counts.show(vertical=True)

-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 3251890  
 trip_distance         | 0        
 RatecodeID            | 3251890  
 store_and_fwd_flag    | 3251890  
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 3251890  
 airport_fee           | 3251890  
 cbd_congestion_fee    | 18939265 



In [17]:
# Estadísticas básicas
df.select(
    "trip_distance", "fare_amount", "tip_amount",
    "total_amount", "passenger_count"
).summary("count", "min", "max", "mean").show()

+-------+----------------+------------------+------------------+-----------------+------------------+
|summary|   trip_distance|       fare_amount|        tip_amount|     total_amount|   passenger_count|
+-------+----------------+------------------+------------------+-----------------+------------------+
|  count|        30137291|          30137291|          30137291|         30137291|          26885401|
|    min|             0.0|           -1807.6|            -300.0|         -1832.85|               0.0|
|    max|       335004.33|         863372.12|            999.99|        863380.37|               9.0|
|   mean|4.78403644773492|17.982493268883573|3.1604352806630422|26.54542605999227|1.3263808488480422|
+-------+----------------+------------------+------------------+-----------------+------------------+



In [18]:
# Filas de muestra
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2025-03-31 08:25:18 |2025-03-31 08:31:33  |1.0            |0.92         |1.0       |N                 |164         |233 

## 7. Taxi Zone Lookup

In [19]:
# Se comprueba si la tabla de referencia existe y se inspecciona
try:
    zones = spark.table("nessie.nyc.taxi_zone_lookup")
    print(f"Registros: {zones.count()}")
    zones.printSchema()
    zones.show(10, truncate=False)
except Exception as e:
    print(f"La tabla aún no se ha ingestado: {e}")

Registros: 265
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 10 rows

